# HuggingFace datasets per language

Scrapes the per-language dataset listings from the HuggingFace Hub.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
html_text = requests.get('https://huggingface.co/languages').text
soup = BeautifulSoup(html_text, 'html.parser')


rows = soup.find_all('tr')

# Initialize lists to store data
iso_codes = []

for row in rows:
    cells = row.find_all('td')
    if len(cells) > 0:  # Exclude header row
        iso_codes.append(cells[1].find('code').get_text(strip=True))

In [ ]:
html_content = requests.get('https://huggingface.co/models?sort=trending')
soup = BeautifulSoup(html_content.text, 'html.parser')

# Find all divs with class="mb-3"
mb3_divs = soup.find_all(class_="mb-3")

# Initialize lists to store results
href_list = []

for div in mb3_divs:    
    # Find all "a" tags within the current mb-3 div
    hrefs = div.find_all('a')
    # Extract href attributes from each "a" tag and append to the list
    href_list.extend([a['href'] for a in hrefs])
models_names = [s.replace('/models?pipeline_tag=', '') for s in href_list]

In [ ]:
languages = ["en", "fr", "de"]  # Example list of languages
task_categories = ["text-to-image", "image-to-text", "document-question-answering"]

In [ ]:
def get_datasets_json(task_categories, language, page=None):
    # URL components
    scheme = "https"
    host = "huggingface.co"
    filename = "/datasets-json"

    # Query parameters
    query_params = {
        "task_categories": f"task_categories:{task_categories}",
        "language": f"language:{language}",
        "sort": "trending"
    }
    if page is not None:
        query_params["p"] = page

    # Construct the full URL
    url = f"{scheme}://{host}{filename}"

    # Send GET request with query parameters
    response = requests.get(url, params=query_params)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse JSON response and return it
        return response.json()
    else:
        # Print error message if request failed
        print("Failed to retrieve data. Status code:", response.status_code)
        return None 


# Define lists of languages and pipeline tags
languages = ["en", "fr", "de"]  # Example list of languages
pipeline_tags = ["text-to-image", "image-to-text", "document-question-answering"]  # Example list of pipeline tags

def get_data_for_combinations(languages, pipeline_tags):
    # Initialize an empty list to store the results
    results = []

    # Iterate over each language and pipeline tag combination
    for language in languages:
        for pipeline_tag in pipeline_tags:
            page = None
            while True:
                # Call the get_models_json function for the current combination and page
                response_data = get_datasets_json(pipeline_tag, language, page=page)
                if response_data:
                    # Append the results to the list
                    results.append({"language": language, "pipeline_tag": pipeline_tag, "response": response_data})
                    # Check if there are more pages
                    if len(response_data['datasets']) == 30:
                        page = 1 if page is None else page + 1
                        print(page)
                    else:
                        break
                else:
                    break  # Exit loop if there's an error

    # Create a DataFrame from the results list
    df = pd.DataFrame(results)

    return df

In [ ]:
result_df = get_data_for_combinations(languages, task_categories)

In [ ]:
result_df['datasets_count'] = result_df['response'].apply(lambda x: len(x['datasets']))

In [ ]:
result_df['sum'] = result_df.groupby(['language', 'pipeline_tag'])['datasets_count'].transform('sum')

In [ ]:
df = result_df.drop_duplicates(subset=['language', 'pipeline_tag'])

In [ ]:
df